# 0.0 Imports

In [29]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 0.2 Carregamento dos dados

In [30]:
df = pd.read_csv("../dataset/processado/df_final.csv")

# 1.0 Descrição dos dados

## 1.1 Renomear colunas

In [31]:
df.columns = (df.columns
            .str.lower()
            .str.replace(" ","_")
)

df.columns

Index(['unnamed:_0', 'row_id', 'order_id', 'order_date', 'ship_date',
       'ship_mode', 'customer_id', 'customer_name', 'segment', 'country',
       'city', 'state', 'postal_code', 'region', 'product_id', 'category',
       'sub-category', 'product_name', 'sales', 'quantity', 'discount',
       'profit', 'person', 'ship_days', 'order_month'],
      dtype='str')

# 1.2 Dimensões

In [32]:
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

Quantidade de linhas: 9986
Quantidade de colunas: 25


## 1.3 Tipos de Dados

In [33]:
df.dtypes

unnamed:_0         int64
row_id             int64
order_id             str
order_date           str
ship_date            str
ship_mode            str
customer_id          str
customer_name        str
segment              str
country              str
city                 str
state                str
postal_code      float64
region               str
product_id           str
category             str
sub-category         str
product_name         str
sales            float64
quantity           int64
discount         float64
profit           float64
person               str
ship_days          int64
order_month          str
dtype: object

In [34]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# 1.4 Check NA

In [35]:
df.isna().sum()

unnamed:_0        0
row_id            0
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_id       0
customer_name     0
segment           0
country           0
city              0
state             0
postal_code      11
region            0
product_id        0
category          0
sub-category      0
product_name      0
sales             0
quantity          0
discount          0
profit            0
person            0
ship_days         0
order_month       0
dtype: int64

In [36]:
df['postal_code'] = df['postal_code'].astype('Int64')

# 2.0 Criação de Variáveis

In [37]:
product_agg = (
    df.groupby('product_name')
    .agg(
        sales=('sales', 'sum'),
        quantity=('quantity', 'sum'),
        profit=('profit', 'sum'),
        avg_discount=('discount', 'mean'),
        avg_ship_days=('ship_days', 'mean'),
        orders=('order_id', 'nunique'),
    )
)

In [38]:
product_agg

,sales,quantity,profit,avg_discount,avg_ship_days,orders
product_name,,,,,,
"""While you Were Out"" Message Book, One Form per Page",25.22,8,10.39,0.133333,4.333333,3
"#10 Gummed Flap White Envelopes, 100/Box",41.30,11,16.77,0.100000,5.750000,4
#10 Self-Seal White Envelopes,108.68,10,52.12,0.050000,4.250000,4
"#10 White Business Envelopes,4 1/8 x 9 1/2",488.91,32,223.12,0.057143,4.285714,7
"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",286.67,37,115.30,0.080000,4.600000,10
...,...,...,...,...,...,...
"iKross Bluetooth Portable Keyboard + Cell Phone Stand Holder + Brush for Apple iPhone 5S 5C 5, 4S 4",477.66,24,115.64,0.080000,5.000000,5
iOttie HLCRIO102 Car Mount,215.89,12,-11.99,0.160000,3.200000,5
iOttie XL Car Mount,223.89,14,-50.37,0.200000,2.000000,2


## 2.1 Lucro

In [39]:
profit = df[['profit', 'product_name']].groupby('product_name').sum()

In [40]:
profit

,profit
product_name,
"""While you Were Out"" Message Book, One Form per Page",10.39
"#10 Gummed Flap White Envelopes, 100/Box",16.77
#10 Self-Seal White Envelopes,52.12
"#10 White Business Envelopes,4 1/8 x 9 1/2",223.12
"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",115.30
...,...
"iKross Bluetooth Portable Keyboard + Cell Phone Stand Holder + Brush for Apple iPhone 5S 5C 5, 4S 4",115.64
iOttie HLCRIO102 Car Mount,-11.99
iOttie XL Car Mount,-50.37


## 2.2 Crescimento

In [41]:
df['year'] = df['order_date'].dt.year

In [42]:
sales_2017 = df[df['year'] == 2017].groupby('product_name')['sales'].sum()
sales_2018 = df[df['year'] == 2018].groupby('product_name')['sales'].sum()

product_agg['growth'] = (
    (sales_2018 - sales_2017) / sales_2017 * 100).round(2)

product_agg['growth'] = product_agg['growth'].fillna(0)


In [43]:
product_agg

,sales,quantity,profit,avg_discount,avg_ship_days,orders,growth
product_name,,,,,,,
"""While you Were Out"" Message Book, One Form per Page",25.22,8,10.39,0.133333,4.333333,3,0.00
"#10 Gummed Flap White Envelopes, 100/Box",41.30,11,16.77,0.100000,5.750000,4,-60.01
#10 Self-Seal White Envelopes,108.68,10,52.12,0.050000,4.250000,4,-74.36
"#10 White Business Envelopes,4 1/8 x 9 1/2",488.91,32,223.12,0.057143,4.285714,7,-64.00
"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",286.67,37,115.30,0.080000,4.600000,10,0.00
...,...,...,...,...,...,...,...
"iKross Bluetooth Portable Keyboard + Cell Phone Stand Holder + Brush for Apple iPhone 5S 5C 5, 4S 4",477.66,24,115.64,0.080000,5.000000,5,-57.89
iOttie HLCRIO102 Car Mount,215.89,12,-11.99,0.160000,3.200000,5,114.29
iOttie XL Car Mount,223.89,14,-50.37,0.200000,2.000000,2,0.00


## 2.3 Volume

In [44]:
quantity = df[['quantity', 'product_name']].groupby('product_name').sum()

In [45]:
quantity

,quantity
product_name,
"""While you Were Out"" Message Book, One Form per Page",8
"#10 Gummed Flap White Envelopes, 100/Box",11
#10 Self-Seal White Envelopes,10
"#10 White Business Envelopes,4 1/8 x 9 1/2",32
"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",37
...,...
"iKross Bluetooth Portable Keyboard + Cell Phone Stand Holder + Brush for Apple iPhone 5S 5C 5, 4S 4",24
iOttie HLCRIO102 Car Mount,12
iOttie XL Car Mount,14


## 2.4 Margem

In [46]:
product_agg['margin'] = (
    product_agg['profit'] / product_agg['sales']
)

In [47]:
product_agg

,sales,quantity,profit,avg_discount,avg_ship_days,orders,growth,margin
product_name,,,,,,,,
"""While you Were Out"" Message Book, One Form per Page",25.22,8,10.39,0.133333,4.333333,3,0.00,0.411975
"#10 Gummed Flap White Envelopes, 100/Box",41.30,11,16.77,0.100000,5.750000,4,-60.01,0.406053
#10 Self-Seal White Envelopes,108.68,10,52.12,0.050000,4.250000,4,-74.36,0.479573
"#10 White Business Envelopes,4 1/8 x 9 1/2",488.91,32,223.12,0.057143,4.285714,7,-64.00,0.456362
"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",286.67,37,115.30,0.080000,4.600000,10,0.00,0.402205
...,...,...,...,...,...,...,...,...
"iKross Bluetooth Portable Keyboard + Cell Phone Stand Holder + Brush for Apple iPhone 5S 5C 5, 4S 4",477.66,24,115.64,0.080000,5.000000,5,-57.89,0.242097
iOttie HLCRIO102 Car Mount,215.89,12,-11.99,0.160000,3.200000,5,114.29,-0.055538
iOttie XL Car Mount,223.89,14,-50.37,0.200000,2.000000,2,0.00,-0.224977


## 2.5 Estabilidade

In [48]:
years = [2015, 2016, 2017, 2018]

sales_year = (
    df[['sales', 'product_name', 'year']].groupby(['product_name', 'year'])
      .sum()
      .unstack(fill_value=0)
)

annual_sales_mean = sales_year.mean(axis=1)

annual_sales_std = sales_year.std(axis=1)

sales_cv = (
    annual_sales_std / annual_sales_mean.replace(0, np.nan)
)

stability_score = (
    1 - sales_cv.rank(pct=True)
) * 100

product_agg['stability_score'] = stability_score

In [49]:
product_agg

,sales,quantity,profit,avg_discount,avg_ship_days,orders,growth,margin,stability_score
product_name,,,,,,,,,
"""While you Were Out"" Message Book, One Form per Page",25.22,8,10.39,0.133333,4.333333,3,0.00,0.411975,4.540541
"#10 Gummed Flap White Envelopes, 100/Box",41.30,11,16.77,0.100000,5.750000,4,-60.01,0.406053,48.540541
#10 Self-Seal White Envelopes,108.68,10,52.12,0.050000,4.250000,4,-74.36,0.479573,16.216216
"#10 White Business Envelopes,4 1/8 x 9 1/2",488.91,32,223.12,0.057143,4.285714,7,-64.00,0.456362,69.351351
"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",286.67,37,115.30,0.080000,4.600000,10,0.00,0.402205,68.918919
...,...,...,...,...,...,...,...,...,...
"iKross Bluetooth Portable Keyboard + Cell Phone Stand Holder + Brush for Apple iPhone 5S 5C 5, 4S 4",477.66,24,115.64,0.080000,5.000000,5,-57.89,0.242097,56.162162
iOttie HLCRIO102 Car Mount,215.89,12,-11.99,0.160000,3.200000,5,114.29,-0.055538,55.459459
iOttie XL Car Mount,223.89,14,-50.37,0.200000,2.000000,2,0.00,-0.224977,4.540541


In [50]:
product_agg.to_csv("../dataset/processado/product_agg.csv")

# 3.0 Normalização

In [23]:
scaler = MinMaxScaler()

cols = [
    'profit',
    'growth',
    'quantity',
    'margin',
    'stability_score'
]

product_agg[cols] = scaler.fit_transform(
    product_agg[cols]
)

In [24]:
product_agg.describe()

,sales,quantity,profit,avg_discount,avg_ship_days,orders,growth,margin,stability_score
count,1850.000000,1850.000000,1850.000000,1850.000000,1850.000000,1850.000000,1850.000000,1850.000000,1850.000000
mean,1240.815919,0.090907,0.265100,0.151463,3.948244,5.397838,0.035071,0.906096,0.497827
std,2793.913407,0.060832,0.023766,0.133274,0.906764,3.132481,0.053753,0.074047,0.290108
min,1.620000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,98.220000,0.051402,0.260862,0.066667,3.428571,3.000000,0.018149,0.871290,0.246741
50%,315.925000,0.084112,0.261878,0.111111,4.000000,5.000000,0.020812,0.915453,0.497827
75%,1205.625000,0.121495,0.264781,0.200000,4.500000,7.000000,0.030549,0.961603,0.748914
max,61599.830000,1.000000,1.000000,0.800000,7.000000,48.000000,1.000000,1.000000,1.000000


In [25]:
product_agg['priority_score'] = (
    product_agg['profit'] * 0.3
    + product_agg['growth'] * 0.25
    + product_agg['quantity'] * 0.2
    + product_agg['margin'] * 0.15
    + product_agg['stability_score'] * 0.1
)

In [26]:
ranking = product_agg['priority_score'].sort_values(ascending=False)

In [27]:
ranking

product_name
Staples                                                                     0.528944
Canon imageCLASS 2200 Advanced Copier                                       0.505544
Staple envelope                                                             0.489372
Easy-staple paper                                                           0.476219
Acco 3-Hole Punch                                                           0.469384
                                                                              ...   
Euro Pro Shark Stick Mini Vacuum                                            0.159169
Okidata B401 Printer                                                        0.149707
Zebra GK420t Direct Thermal/Thermal Transfer Printer                        0.149538
Bush Westfield Collection Bookcases, Dark Cherry Finish, Fully Assembled    0.118029
Eureka Disposable Bags for Sanitaire Vibra Groomer I Upright Vac            0.088395
Name: priority_score, Length: 1850, dtype: float64